In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import nbformat

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


# Read merged file
hf_df = pd.read_csv("cleaned_files/heart_failure_merged.csv")
# Creating a copy of the data to work on, so the original data is not modified
df = hf_df.copy()


### ***Q1.Which cci_score groups should be prioritized when designing a follow-up program***

**Reasoning:**

<i>Comparing outcomes across CCI score groups helps identify which levels of comorbidity burden have higher observed readmission or mortality rates. Those groups may be candidates for closer discharge planning and follow-up. Patient counts should be considered alongside outcome rates, since rates from small groups can be unstable.<i>

In [ ]:
df["cci_score"] = pd.to_numeric(df["cci_score"], errors="coerce")

# Short-term adverse outcomes
for col in ["readmission_28d", "mortality_28d", "readmission_3mo", "death_3mo"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df["poor_outcome"] = df[["readmission_28d", "mortality_28d", "readmission_3mo", "death_3mo"]].max(axis=1)

# -----------------------------
#  CCI groups
# -----------------------------
df["cci_group"] = pd.cut(
    df["cci_score"],
    bins=[-1, 1, 3, 5, 10, 20],
    labels=["0-1", "2-3", "4-5", "6-10", "11+"],
    right=True
)

# -----------------------------
# Summarize risk by CCI group
# -----------------------------
outcome_cols = ["readmission_28d", "mortality_28d", "readmission_3mo", "death_3mo", "poor_outcome"]

cci_summary = (
    df.groupby("cci_group")[outcome_cols]
      .mean()
      .mul(100)
      .round(2)
)

print("Outcome rates by CCI group (%)")
print(cci_summary)

# -----------------------------
#  Identify high-priority groups
# -----------------------------
priority_groups = cci_summary.sort_values("poor_outcome", ascending=False).reset_index()

# Define a practical follow-up priority threshold
high_priority = priority_groups[priority_groups["poor_outcome"] >= priority_groups["poor_outcome"].median()].copy()
print("\nHigh-priority CCI groups for follow-up program:")
print(high_priority)



In [ ]:
# -----------------------------
# Plot: outcome rates by CCI group
# -----------------------------
plot_df = cci_summary.reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(data=plot_df, x="cci_group", y="poor_outcome", palette="Reds_d")
plt.title("Poor short-term outcome rate by CCI group")
plt.xlabel("CCI group")
plt.ylabel("Poor outcome rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# -----------------------------
# Plot :count patients in each CCI group
# -----------------------------
count_df = df["cci_group"].value_counts().sort_index().reset_index()
count_df.columns = ["cci_group", "n_patients"]

print("\nPatient counts by CCI group:")
print(count_df)

# Combined plot: counts + poor outcome rate
fig, ax1 = plt.subplots(figsize=(10, 6))

color1 = "#4C72B0"
color2 = "#DD8452"

ax1.bar(count_df["cci_group"], count_df["n_patients"], color=color1, alpha=0.7, label="Patients")
ax1.set_xlabel("CCI group")
ax1.set_ylabel("Number of patients", color=color1)
ax1.tick_params(axis="y", labelcolor=color1)

ax2 = ax1.twinx()
ax2.plot(plot_df["cci_group"], plot_df["poor_outcome"], color=color2, marker="o", linewidth=2, label="Poor outcome %")
ax2.set_ylabel("Poor outcome rate (%)", color=color2)
ax2.tick_params(axis="y", labelcolor=color2)

plt.title("CCI group size vs poor outcome burden")
plt.tight_layout()
plt.show()

<i>**Key Insights:**
1. Poor short-term outcome rates appear to rise with CCI score: approximately 24% for scores 0–1, 27% for 2–3, 48% for 4–5, and 100% for 6–10. This suggests the 4–10 groups may merit closer review when planning follow-up.
2. The 2–3 group is the largest and, despite its lower rate, may account for substantial follow-up workload.A practical approach is to review higher-CCI patients more closely while planning scalable support for the large 2–3 group which is 6-10 CCI-group.<i>

### ***Q2.Do patients with recorded chronic kidney disease have higher readmission rates, suggesting a need to review their follow-up coordination?***

 **Reasoning:**
 
 <i>This analysis compares readmission rates at 28 days, 3 months, and 6 months between patients with and without recorded moderate-to-severe CKD. CKD may add complexity to heart-failure care, so higher observed readmission rates in that group could make it a useful group for reviewing discharge instructions, medication coordination, and follow-up arrangements. Comparing several time windows helps show whether the difference is concentrated soon after discharge or persists over time.<i>

In [ ]:
# CKD indicator: mod_severe_ckd
df["ckd"] = pd.to_numeric(df["mod_severe_ckd"], errors="coerce").fillna(0)

# Readmission outcomes
for col in ["readmission_28d", "readmission_3mo", "readmission_6mo"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Short-term poor outcome indicator
df["poor_outcome"] = df[["readmission_28d", "readmission_3mo", "readmission_6mo"]].max(axis=1)

# -----------------------------
# Compare readmission rates by CKD status
# -----------------------------
outcomes = ["readmission_28d", "readmission_3mo", "readmission_6mo", "poor_outcome"]

comparison = []
for outcome in outcomes:
    no_ckd = df.loc[df["ckd"] == 0, outcome]
    yes_ckd = df.loc[df["ckd"] == 1, outcome]

    comparison.append({
        "Outcome": outcome,
        "No CKD (%)": no_ckd.mean() * 100,
        "CKD (%)": yes_ckd.mean() * 100,
        "Difference (%)": (yes_ckd.mean() - no_ckd.mean()) * 100
    })

comparison_df = pd.DataFrame(comparison)
print(comparison_df.round(2))


In [ ]:
# -----------------------------
#  Plot: readmission rate by CKD status
# -----------------------------
plot_df = comparison_df[["Outcome", "No CKD (%)", "CKD (%)"]].copy()
plot_df = plot_df.melt(id_vars="Outcome", var_name="CKD_Status", value_name="Rate")

# Convert labels
plot_df["CKD_Status"] = plot_df["CKD_Status"].replace({
    "No CKD (%)": "No CKD",
    "CKD (%)": "CKD"
})

plt.figure(figsize=(10, 6))
sns.barplot(data=plot_df, x="Outcome", y="Rate", hue="CKD_Status", palette=["#4C72B0", "#DD8452"])
plt.title("Readmission rate by CKD status")
plt.xlabel("Outcome time point")
plt.ylabel("Rate (%)")
plt.ylim(0, max(plot_df["Rate"].max() * 1.2, 20))
plt.xticks(rotation=0)
plt.legend(title="CKD status")
plt.tight_layout()
plt.show()

# -----------------------------
#   patient counts
# -----------------------------
ckd_counts = df["ckd"].value_counts().rename(index={0: "No CKD", 1: "CKD"})
print("\nCKD counts:")
print(ckd_counts)

<i>**Key Insights:**
 1. Patients recorded with moderate-to-severe CKD have higher readmission rates at each interval: roughly 9% vs. 6% at 28 days, 31% vs. 23% at 3 months, and 46% vs. 36% at 6 months.
2. The difference grows over time, suggesting CKD may be a useful flag when reviewing follow-up coordination beyond the immediate post-discharge period.<i>

### ***Q3.Do patients with diabetes and congestive heart failure have higher readmission rates than patients with either condition alone, and should that combination receive focused review?***


**Reasoning:**

 <i> This analysis helps to analyse if patients with diabetes and heart failure have higher readmission rates than heart failure patients without diabetes.
 Diabetes can add to the complexity of managing heart failure. Comparing 28-day, 3-month, and 6-month readmission rates between these two groups helps show whether heart-failure patients with diabetes experience more readmissions, and whether any difference persists over time. If their rates are higher, the group may be worth prioritizing for a review of discharge planning and follow-up coordination<i>

In [ ]:
# Diabetes indicator
df["diabetes"] = pd.to_numeric(df["diabetes"], errors="coerce").fillna(0)

df["chf_group"] = 1  # all patients are in the dataset are grouped as CHF 

# Readmission outcomes
for col in ["readmission_28d", "readmission_3mo", "readmission_6mo"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Make a 4-group comparison
df["group"] = np.where(
    (df["diabetes"] == 1),
    "CHF + diabetes",
    "CHF without diabetes"
)

# -----------------------------
#  Calculate readmission rates by group
# -----------------------------
outcomes = ["readmission_28d", "readmission_3mo", "readmission_6mo"]

summary = []
for outcome in outcomes:
    result = df.groupby("group")[outcome].mean().mul(100).reset_index()
    result.columns = ["group", outcome]
    summary.append(result)

rate_table = summary[0]
for r in summary[1:]:
    rate_table = rate_table.merge(r, on="group", how="outer")

print("Readmission rates by group:")
print(rate_table.round(2))


In [ ]:
# -----------------------------
#  Plot grouped bar chart
# -----------------------------
plot_df = rate_table.melt(id_vars="group", var_name="Outcome", value_name="Rate")

plt.figure(figsize=(10, 6))
sns.barplot(data=plot_df, x="Outcome", y="Rate", hue="group", palette=["#4C72B0", "#DD8452"])
plt.title("Readmission rates by diabetes status in CHF cohort")
plt.xlabel("Follow-up period")
plt.ylabel("Readmission rate (%)")
plt.xticks(rotation=0)
plt.legend(title="Patient group")
plt.tight_layout()
plt.show()

<i>**Key Insights:**
1. Patients with diabetes have higher readmission rates at all three follow-up points: about 10% vs. 6% at 28 days, 31% vs. 23% at 3 months, and 46% vs. 36% at 6 months.
2. The difference grows over time, from roughly 4 percentage points at 28 days to 10 points at 6 months. This suggests the CHF-plus-diabetes group may merit focused review of follow-up coordination.
3. Readmission rates rise in both groups, so ongoing follow-up may be important for the whole CHF cohort, not only patients with diabetes.<i>

### ***Q4.Which combinations of recorded conditions occur most often among readmitted patients, helping define groups for a pilot program?***

 **Reasoning:**

 <i>This analysis identifies patients who had at least one recorded readmission within 28 days, 3 months, or 6 months, then counts the combinations of recorded conditions among them. This shows which clinical profiles are most frequently represented in the readmitted group and can help generate practical groups for a focused follow-up pilot. <i>

In [ ]:
readmission_cols = ["readmission_28d", "readmission_3mo", "readmission_6mo"]
for col in readmission_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# A patient is considered readmitted if they had any readmission during follow-up
df["readmitted_any"] = df[readmission_cols].max(axis=1)

readmitted = df[df["readmitted_any"] == 1].copy()

# -----------------------------
# Select key comorbidity columns
# -----------------------------
condition_cols = [
    "diabetes",
    "mod_severe_ckd",
    "copd",
    "dementia",
    "cerebrovascular_disease",
    "connective_tissue_disease",
    "peptic_ulcer_disease",
    "malignant_lymphoma",
    "solid_tumor",
    "liver_disease",
    "aids"
]

for col in condition_cols:
    if col in df.columns:
        readmitted[col] = pd.to_numeric(readmitted[col], errors="coerce").fillna(0)

# -----------------------------
#  Build condition combinations
# -----------------------------
# Keep only condition columns with value 1 = present
readmitted["condition_set"] = readmitted[condition_cols].astype(int).apply(
    lambda row: tuple(col for col, val in zip(condition_cols, row) if val == 1),
    axis=1
)

# Count combinations
combo_counts = readmitted["condition_set"].value_counts().reset_index()
combo_counts.columns = ["condition_combo", "count"]

# Remove empty combinations if any
combo_counts = combo_counts[combo_counts["condition_combo"].astype(str) != "()"]

# Keep the most common combinations for interpretation
top_combinations = combo_counts.head(10).copy()
print("Top condition combinations among readmitted patients:")
print(top_combinations)



In [ ]:
# -----------------------------
# Prepare labels for plotting
# -----------------------------
top_combinations["label"] = top_combinations["condition_combo"].apply(
    lambda x: ", ".join(x) if len(x) > 0 else "No recorded condition"
)

# -----------------------------
# Plot: top combinations
# -----------------------------
plt.figure(figsize=(12, 7))
plt.barh(top_combinations["label"][::-1], top_combinations["count"][::-1], color="steelblue")
plt.title("Most common condition combinations among readmitted patients")
plt.xlabel("Number of readmitted patients")
plt.ylabel("Condition combination")
plt.tight_layout()
plt.show()

<i>**Key Insights:**
 1. Among readmitted patients, the most frequent exact recorded profiles are diabetes alone (about 90 patients) and moderate-to-severe CKD alone (about 88).
 2. The diabetes + CKD combination is also common, at about 61 patients. COPD alone (about 40) and dementia alone (about 36) appear next.
 3. Counts drop off for the remaining combinations, so diabetes and CKD profiles stand out as practical candidates to examine when shaping a follow-up pilot.<i>

### ***Q5.Do higher NYHA classes identify patients for whom the existing discharge and follow-up process should be reviewed?***

 **Reasoning :**

 <i>Comparing readmission and mortality outcomes across severity groups can help identify whether patients with greater clinical burden may need closer review of discharge planning, early follow-up, or monitoring. If poor outcomes are more frequent in higher-severity groups, those groups could be considered for focused process review. <i>

In [ ]:
for col in ["readmission_28d", "readmission_3mo", "readmission_6mo", "mortality_28d", "death_3mo"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df["poor_outcome"] = df[["readmission_28d", "mortality_28d", "readmission_3mo", "death_3mo", "readmission_6mo"]].max(axis=1)


# Build NYHA-like severity 

df["oxygen_yes"] = df["oxygen_use"].fillna("").astype(str).str.contains("oxygen|vent|support", case=False, na=False).astype(int)
df["resp_support_yes"] = df["resp_support"].fillna("").astype(str).str.strip().ne("").astype(int)
df["discharge_day"] = pd.to_numeric(df["discharge_day"], errors="coerce").fillna(0)
df["gcs"] = pd.to_numeric(df["gcs"], errors="coerce").fillna(15)
df["bnp"] = pd.to_numeric(df["bnp"], errors="coerce").fillna(0)

df["nyha_proxy_score"] = (
    2 * df["oxygen_yes"]
    + 2 * df["resp_support_yes"]
    + 1 * (df["discharge_day"] > 7).astype(int)
    + 1 * (df["gcs"] < 15).astype(int)
    + 1 * (df["bnp"] > 400).astype(int)
)

# Map score to NYHA-style groups
bins = [-1, 0, 1, 3, 6, 99]
labels = ["I", "II", "III", "IV", "IV+"]
df["nyha_group"] = pd.cut(df["nyha_proxy_score"], bins=bins, labels=labels, right=True)


# Compare outcome rates by NYHA-style class
# -----------------------------
summary = (
    df.groupby("nyha_group")
      .agg(
          patients=("patient_id", "count"),
          readmission_28d=("readmission_28d", "mean"),
          mortality_28d=("mortality_28d", "mean"),
          readmission_3mo=("readmission_3mo", "mean"),
          death_3mo=("death_3mo", "mean"),
          readmission_6mo=("readmission_6mo", "mean"),
          poor_outcome=("poor_outcome", "mean")
      )
      .mul(100)
      .round(2)
)

print("NYHA-style severity summary (proxy classes):")
print(summary)

# Identify classes above median poor-outcome rate for review
review_threshold = summary["poor_outcome"].median()
review_classes = summary[summary["poor_outcome"] >= review_threshold].index.tolist()

print("\nProxy classes above the median poor-outcome rate:")
print(review_classes)



In [ ]:
# -----------------------------
# Plot outcome burden by class
# -----------------------------
plot_df = summary[["poor_outcome"]].reset_index().rename(
    columns={"nyha_group": "NYHA_proxy_class", "poor_outcome": "Poor outcome rate (%)"}
)

plt.figure(figsize=(10, 6))
sns.barplot(data=plot_df, x="NYHA_proxy_class", y="Poor outcome rate (%)", palette="Reds_d")
plt.title("Poor short-term outcome rate by NYHA-style severity class")
plt.xlabel("NYHA-style class")
plt.ylabel("Poor outcome rate (%)")
plt.tight_layout()
plt.show()

<i>**Key Insights :**
 1. The highest poor-outcome rate is in Class II (about 46%); Class I is lowest at roughly 30%.
 2. Rates for Classes III, IV, and IV+ are around 39–44%. They do not rise steadily with class, so this chart does not show a clear pattern of progressively worse outcomes at higher levels.
 3. All groups have notable poor-outcome rates.<i>


## ***Q6.Do higher Killip grades identify groups with higher short-term adverse outcome rates that warrant closer process review?***

 **Reasoning:**

 <i>Killip class reflects the severity of a patient’s acute heart-failure presentation. Comparing short-term readmission, mortality, and composite poor-outcome rates across severity groups can show whether patients with greater clinical burden may need closer review of discharge planning, monitoring, or follow-up.<i>

In [ ]:
for col in ["readmission_28d", "readmission_3mo", "readmission_6mo", "mortality_28d", "death_3mo"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df["poor_outcome"] = df[["readmission_28d", "mortality_28d", "readmission_3mo", "death_3mo", "readmission_6mo"]].max(axis=1)


#  Killip-style severity 
# -----------------------------

df["oxygen_yes"] = df["oxygen_use"].fillna("").astype(str).str.contains("oxygen|vent|support", case=False, na=False).astype(int)
df["resp_support_yes"] = df["resp_support"].fillna("").astype(str).str.strip().ne("").astype(int)
df["discharge_day"] = pd.to_numeric(df["discharge_day"], errors="coerce").fillna(0)
df["gcs"] = pd.to_numeric(df["gcs"], errors="coerce").fillna(15)
df["bnp"] = pd.to_numeric(df["bnp"], errors="coerce").fillna(0)

df["killip_proxy_score"] = (
    2 * df["oxygen_yes"]
    + 2 * df["resp_support_yes"]
    + 1 * (df["discharge_day"] > 7).astype(int)
    + 1 * (df["gcs"] < 15).astype(int)
    + 1 * (df["bnp"] > 400).astype(int)
)

# Map to Killip-style classes
bins = [-1, 0, 1, 3, 6, 99]
labels = ["Killip I", "Killip II", "Killip III", "Killip IV", "Killip IV+"]
df["killip_group"] = pd.cut(df["killip_proxy_score"], bins=bins, labels=labels, right=True)


# Compare short-term adverse outcome rates by Killip group
# -----------------------------
killip_summary = (
    df.groupby("killip_group")
      .agg(
          patients=("patient_id", "count"),
          readmission_28d=("readmission_28d", "mean"),
          mortality_28d=("mortality_28d", "mean"),
          readmission_3mo=("readmission_3mo", "mean"),
          death_3mo=("death_3mo", "mean"),
          readmission_6mo=("readmission_6mo", "mean"),
          poor_outcome=("poor_outcome", "mean")
      )
      .mul(100)
      .round(2)
)

print("Short-term adverse outcome rates by Killip-style group (%)")
print(killip_summary)

# Review threshold: groups at or above median poor-outcome rate

review_threshold = killip_summary["poor_outcome"].median()
review_groups = killip_summary[killip_summary["poor_outcome"] >= review_threshold].index.tolist()
print("\nKillip groups above the median poor-outcome rate:")
print(review_groups)


In [ ]:

# Plot poor outcome rate by Killip group
# -----------------------------
plot_df = killip_summary[["poor_outcome"]].reset_index().rename(
    columns={"killip_group": "Killip_group", "poor_outcome": "Poor outcome rate (%)"}
)

plt.figure(figsize=(10, 6))
sns.barplot(data=plot_df, x="Killip_group", y="Poor outcome rate (%)", palette="Blues_d")
plt.title("Poor short-term outcome rate by Killip-style class")
plt.xlabel("Killip class")
plt.ylabel("Poor outcome rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

<i>**Key Insights:**
 1. Killip II has the highest poor-outcome rate at about 46%; Killip I is lowest at roughly 30%.
 2. Rates for Killip III, IV, and IV+ are around 39–44%. They do not increase steadily with class, so the chart does not show a clear dose-response pattern.
 3. All groups have notable poor-outcome rates.<i>


### ***Q7.Does recorded myocardial infarction status identify a group with higher 28-day readmission rates?***

 **Reasoning:**

  <i> Myocardial infarction is a major clinical risk factor for recurrent heart-failure events. If the MI group has a higher 28-day readmission rate, it indicates that MI history should be considered when prioritizing discharge planning and early follow-up.<i>

In [ ]:
# Acquire MI status from the merged HF dataset 
# -----------------------------

if "myocardial_infarction" in hf_df.columns:
    df = hf_df.copy()
else:
    cardiac = pd.read_csv("cleaned_files/cardiac_cleaned.csv")
    df = hf_df.merge(cardiac[["patient_id", "myocardial_infarction"]], on="patient_id", how="left")


#  variables
# -----------------------------
df["readmission_28d"] = pd.to_numeric(df["readmission_28d"], errors="coerce").fillna(0)
df["myocardial_infarction"] = pd.to_numeric(df["myocardial_infarction"], errors="coerce").fillna(0)

df["mi_status"] = df["myocardial_infarction"].map({0: "No MI", 1: "MI"})

# -----------------------------
# Compare 28-day readmission rates by MI status
# -----------------------------
summary = (
    df.groupby("mi_status")["readmission_28d"]
      .mean()
      .mul(100)
      .round(2)
      .reset_index()
      .rename(columns={"readmission_28d": "28-day readmission rate (%)"})
)

print("28-day readmission rate by myocardial infarction status:")
print(summary)


In [ ]:
# -----------------------------
# Plot: readmission by MI status
# -----------------------------
plt.figure(figsize=(8, 5))
sns.barplot(data=summary, x="mi_status", y="28-day readmission rate (%)", palette=["#4C72B0", "#DD8452"])
plt.title("28-day readmission rate by myocardial infarction status")
plt.xlabel("Myocardial infarction status")
plt.ylabel("28-day readmission rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

<i>**Key Insights :**
 1. The 28-day readmission rate is higher for patients with recorded MI: about 8.4%, compared with 6.9% for patients recorded as having no MI.
 2. That is a difference of roughly 1.5 percentage points. It suggests MI status may be worth considering in early discharge and follow-up review, but the chart alone does not show whether the difference is statistically reliable or caused by MI.

### ***Q8.Do heart failure types (Left, Right, Both) differ in readmission rates enough to justify reviewing whether follow-up processes meet each group’s needs?***

 **Reasoning:**

 <i>This analysis compares 28-day, 3-month, and 6-month readmission rates among patients recorded as having left-, right-, or both-sided heart failure. Differences across these groups may indicate that some patient subtypes warrant a closer review of discharge planning and follow-up, such as whether monitoring, education, and appointment timing meet their needs. Looking across several time windows helps show whether any difference is early or persists over time<i>

In [ ]:
# Read only the merged HF dataset
hf_df = pd.read_csv("cleaned_files/heart_failure_merged.csv")

# Keep the working data
df = hf_df.copy()

# Clean heart failure type labels
df["heart_failure_type"] = df["heart_failure_type"].astype(str).str.strip().str.lower()

# Keep only the clinically relevant groups
valid_types = ["left", "right", "both"]
df = df[df["heart_failure_type"].isin(valid_types)].copy()

# Clean readmission columns
for col in ["readmission_28d", "readmission_3mo", "readmission_6mo"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Any readmission flag
df["readmitted_any"] = df[["readmission_28d", "readmission_3mo", "readmission_6mo"]].max(axis=1)

# Summarize by heart-failure type
summary = (
    df.groupby("heart_failure_type")
      .agg(
          patients=("patient_id", "count"),
          readmission_28d=("readmission_28d", "mean"),
          readmission_3mo=("readmission_3mo", "mean"),
          readmission_6mo=("readmission_6mo", "mean"),
          any_readmission=("readmitted_any", "mean")
      )
      .mul(100)
      .round(2)
      .reset_index()
)

print("Readmission rate by heart failure type (%)")
print(summary)


In [ ]:
# Prepare plot data
plot_df = summary.melt(
    id_vars="heart_failure_type",
    value_vars=["readmission_28d", "readmission_3mo", "readmission_6mo"],
    var_name="Follow_up_window",
    value_name="Rate"
)

plot_df["Follow_up_window"] = plot_df["Follow_up_window"].str.replace("_", " ").str.title()

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(
    data=plot_df,
    x="Follow_up_window",
    y="Rate",
    hue="heart_failure_type",
    palette=["#4C72B0", "#55A868", "#C44E52"]
)
plt.title("Readmission rate by heart failure type")
plt.xlabel("Follow-up window")
plt.ylabel("Readmission rate (%)")
plt.xticks(rotation=0)
plt.legend(title="HF type")
plt.tight_layout()
plt.show()

<i>**Key Insights :**
 1. Readmission rates rise from 28 days to 6 months for all three heart-failure types.
 2. The Both group has the highest rate at every interval: about 8% at 28 days, 27% at 3 months, and 42% at 6 months.
 3. At 3 and 6 months, Right-sided heart failure has higher rates than Left-sided heart failure (about 20% vs. 17% and 35% vs. 27%, respectively). At 28 days, Left is slightly higher than Right.<i>

### ***Q9.Among patients with an LVEF measurement, do readmission rates differ across LVEF groups, and would collecting LVEF more consistently improve analysis?***

**Reasoning:**

<i>Among patients with an LVEF measurement, comparing 28-day, 3-month, and 6-month readmission rates across HFrEF, HFmrEF, and HFpEF groups can show whether observed readmission patterns differ by LVEF category. Reporting how often LVEF is missing also shows how much of the cohort is excluded from that comparison. More consistent LVEF measurement could increase the number of patients available in each group and make comparisons more informative; if measurement is more likely for certain kinds of patients, missingness could also bias the results<i>

In [ ]:
#  LVEF variable
# -----------------------------
df["lvef"] = pd.to_numeric(df["lvef"], errors="coerce")

# Keep only patients with an LVEF measurement
analysis_df = df.dropna(subset=["lvef"]).copy()

# Define clinically meaningful EF groups
# HFrEF: < 40
# HFmrEF: 40 to 49
# HFpEF: >= 50
analysis_df["lvef_group"] = pd.cut(
    analysis_df["lvef"],
    bins=[-1, 39, 49, 100],
    labels=["HFrEF (<40)", "HFmrEF (40-49)", "HFpEF (>=50)"],
    right=True
)


#  readmission variables
# -----------------------------
for col in ["readmission_28d", "readmission_3mo", "readmission_6mo"]:
    analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce").fillna(0)

# -----------------------------
# Summarize readmission rates by LVEF group
# -----------------------------
summary = (
    analysis_df.groupby("lvef_group")
      .agg(
          patients=("patient_id", "count"),
          readmission_28d=("readmission_28d", "mean"),
          readmission_3mo=("readmission_3mo", "mean"),
          readmission_6mo=("readmission_6mo", "mean")
      )
      .mul(100)
      .round(2)
      .reset_index()
)

print("Readmission rates by LVEF group (%)")
print(summary)


In [ ]:
# -----------------------------
# Plot: readmission rate by LVEF group
# -----------------------------
plot_df = summary.melt(
    id_vars="lvef_group",
    value_vars=["readmission_28d", "readmission_3mo", "readmission_6mo"],
    var_name="Follow_up_window",
    value_name="Rate"
)

plot_df["Follow_up_window"] = plot_df["Follow_up_window"].str.replace("_", " ").str.title()

plt.figure(figsize=(10, 6))
sns.barplot(
    data=plot_df,
    x="Follow_up_window",
    y="Rate",
    hue="lvef_group",
    palette=["#4C72B0", "#55A868", "#C44E52"]
)
plt.title("Readmission rates by LVEF group")
plt.xlabel("Follow-up window")
plt.ylabel("Readmission rate (%)")
plt.xticks(rotation=0)
plt.legend(title="LVEF group")
plt.tight_layout()
plt.show()

<i>**Key Insights :**
1. Readmission rates increase from 28 days to 6 months across all LVEF groups.
2. HFrEF (<40%) has the highest rates at 3 months (about 26.5%) and 6 months (about 43%). HFpEF (≥50%) is slightly highest at 28 days (about 7%).
3. At 6 months, the rate is about 43% for HFrEF, 38% for HFpEF, and 36% for HFmrEF. <i>

### ***Q10.Which combinations of cardiac findings and prior conditions have the highest observed readmission rates, and are those groups large enough for a targeted follow-up pilot?***

 **Reasoning:**

 <i>This analysis combines recorded cardiac findings and prior conditions into patient risk profiles, then compares each profile’s observed readmission rate with its patient count. This helps identify profiles that may be practical candidates for a targeted follow-up pilot: a high rate suggests elevated observed readmission burden, while a sufficiently large group may be more feasible to reach and evaluatec.<i>

In [ ]:
# -----------------------------
# clinical variables
# -----------------------------
numeric_cols = [
    "readmission_28d", "readmission_3mo", "readmission_6mo",
    "myocardial_infarction", "congestive_heart_failure", "peripheral_vascular_disease",
    "diabetes", "mod_severe_ckd", "copd", "dementia", "cerebrovascular_disease",
    "lvef", "nyha_cardiac_function_classification", "killip_grade"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# LVEF group for clinical context
df["lvef_group"] = pd.cut(
    df["lvef"],
    bins=[-1, 39, 49, 100],
    labels=["HFrEF", "HFmrEF", "HFpEF"],
    right=True
)

# Heart failure type label
df["heart_failure_type"] = df["heart_failure_type"].fillna("unknown").astype(str).str.lower()

# Binary any-readmission flag
df["readmitted_any"] = (df[["readmission_28d", "readmission_3mo", "readmission_6mo"]].max(axis=1) > 0).astype(int)

# -----------------------------
# Create a combined risk profile label
# -----------------------------
def build_risk_combo(row):
    flags = []

    # Cardiac findings
    if row["myocardial_infarction"] == 1:
        flags.append("MI")
    if row["congestive_heart_failure"] == 1:
        flags.append("CHF")
    if row["peripheral_vascular_disease"] == 1:
        flags.append("PVD")
    if row["lvef_group"] == "HFrEF":
        flags.append("HFrEF")
    elif row["lvef_group"] == "HFmrEF":
        flags.append("HFmrEF")
    elif row["lvef_group"] == "HFpEF":
        flags.append("HFpEF")

    # Prior conditions
    if row["diabetes"] == 1:
        flags.append("DM")
    if row["mod_severe_ckd"] == 1:
        flags.append("CKD")
    if row["copd"] == 1:
        flags.append("COPD")
    if row["dementia"] == 1:
        flags.append("Dementia")
    if row["cerebrovascular_disease"] == 1:
        flags.append("CVD")

    # Optional subtype
    if row["heart_failure_type"] in ["left", "right", "both"]:
        flags.append(row["heart_failure_type"].title())

    return " + ".join(flags) if flags else "No major flags"

df["risk_combo"] = df.apply(build_risk_combo, axis=1)

# -----------------------------
# Summarize readmission by risk combination
# -----------------------------
combo_summary = (
    df.groupby("risk_combo")
      .agg(
          n_patients=("patient_id", "count"),
          readmission_28d=("readmission_28d", "mean"),
          readmission_3mo=("readmission_3mo", "mean"),
          readmission_6mo=("readmission_6mo", "mean"),
          any_readmission=("readmitted_any", "mean")
      )
      .reset_index()
)

# Keep only combinations with meaningful sample size for pilot planning
pilot_threshold = 20
pilot_candidates = combo_summary[combo_summary["n_patients"] >= pilot_threshold].copy()

# Rank by any readmission rate
pilot_candidates = pilot_candidates.sort_values("any_readmission", ascending=False).head(10)

print("Pilot-ready combinations with highest readmission rates:")
print(pilot_candidates.round(3))



In [ ]:
# -----------------------------
#  Plot: top combinations
# -----------------------------
plot_df = pilot_candidates.sort_values("any_readmission", ascending=False).head(8).copy()

plt.figure(figsize=(12, 7))
bars = plt.barh(
    plot_df["risk_combo"][::-1],
    plot_df["any_readmission"][::-1] * 100,
    color="steelblue"
)

plt.title("Highest readmission risk combinations for pilot targeting")
plt.xlabel("Any readmission rate (%)")
plt.ylabel("Risk combination")

# Add patient count labels
for bar, n in zip(bars, plot_df["n_patients"][::-1]):
    plt.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"n={n}",
        va="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

<i>**Key Insights :**
 1. The highest observed readmission rate is for CHF + HFpEF + dementia + Both, at about 53% among 55 patients.
 2. Several other plotted profiles also have high rates, around 42–49%. Many include HFpEF and the Both heart-failure type, suggesting these profiles are worth examining for a follow-up pilot.
 3. CHF + HFpEF + Both is the largest plotted group (466 patients) and has a lower, but still substantial, readmission rate of about 40%. It may represent a practical outreach group because of its size.
 4. The HFpEF + CVD + Both group has only 23 patients, so its rate is less stable. The chart’s minimum group-size cutoff is 20, which still permits relatively small groups.<i>

### ***Q. bnp vs readmission/mortality correlation.***

**Reasoning:**

<i>BNP is a biomarker of cardiac wall stress and HF severity. Patients with higher BNP are expected to have more advanced heart failure and worse short-term outcomes.If BNP rises across the categories from Low to High, the analysis supports using BNP as a risk marker for readmission and mortality.- Stronger positive association with poor outcome suggests that BNP should be considered in discharge planning and follow-up prioritization.<i>

In [ ]:
for col in [
    "bnp", "readmission_28d", "readmission_3mo", "readmission_6mo",
    "mortality_28d", "death_3mo"
]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Create a combined adverse outcome flag
df["poor_outcome"] = df[[
    "readmission_28d", "readmission_3mo", "readmission_6mo",
    "mortality_28d", "death_3mo"
]].max(axis=1)

# -----------------------------
#  BNP severity categories
# -----------------------------

df["bnp_group"] = pd.cut(
    df["bnp"],
    bins=[-1, 100, 400, np.inf],
    labels=["Low (<100)", "Moderate (100-400)", "High (>400)"],
    right=True
)


#  Summarize outcomes by BNP category
# -----------------------------
outcome_cols = [
    "readmission_28d", "readmission_3mo", "readmission_6mo",
    "mortality_28d", "death_3mo", "poor_outcome"
]

summary = (
    df.groupby("bnp_group")[outcome_cols]
      .mean()
      .mul(100)
      .round(2)
      .reset_index()
)

print("Outcome rates by BNP category (%)")
print(summary)



In [ ]:

# Plot: BNP category vs outcomes
# -----------------------------
plot_df = summary.melt(
    id_vars="bnp_group",
    value_vars=["readmission_28d", "readmission_3mo", "readmission_6mo",
                "mortality_28d", "death_3mo", "poor_outcome"],
    var_name="Outcome",
    value_name="Rate_percent"
)

plt.figure(figsize=(12, 7))
sns.barplot(data=plot_df, x="Outcome", y="Rate_percent", hue="bnp_group", palette="Reds")
plt.title("Readmission and mortality by BNP severity category")
plt.xlabel("Outcome")
plt.ylabel("Rate (%)")
plt.xticks(rotation=25)
plt.legend(title="BNP category")
plt.tight_layout()
plt.show()



In [ ]:
# -----------------------------
#  Continuous BNP vs 28-day readmission
# -----------------------------
# Only use rows with some variation in BNP
scatter_df = df[["bnp", "readmission_28d", "mortality_28d"]].dropna()

plt.figure(figsize=(10, 5))
sns.regplot(data=scatter_df, x="bnp", y="readmission_28d", scatter_kws={"alpha": 0.5})
plt.title("BNP vs 28-day readmission rate")
plt.xlabel("BNP")
plt.ylabel("28-day readmission (0/1)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.regplot(data=scatter_df, x="bnp", y="mortality_28d", scatter_kws={"alpha": 0.5})
plt.title("BNP vs 28-day mortality")
plt.xlabel("BNP")
plt.ylabel("28-day mortality (0/1)")
plt.tight_layout()
plt.show()

# -----------------------------
# Correlation summary
# -----------------------------
corr_df = scatter_df[["bnp", "readmission_28d", "mortality_28d"]].corr().round(3)
print("\nCorrelation of BNP with outcomes:")
print(corr_df)


<i>**Key Insights :**
 1. Readmission rates are broadly similar across BNP categories; there is no clear pattern of higher BNP consistently corresponding to higher readmission.
 2. Low BNP has the highest observed 28-day and 6-month readmission rates (about 8% and 40%). High BNP is slightly higher for 3-month readmission and the mortality measures, but those mortality rates are low.
 3. The combined poor-outcome rates are close, around 40–42%, so this chart alone does not show a strong difference in overall outcomes by BNP category.
 4. Rates increase over time across all BNP groups, especially for readmission.